In [2]:
import math
from pathlib import Path

import numpy as np
import rasterio
from rasterio.transform import Affine
import rvt.vis


# =========================
# INPUT
# =========================
dem_path = "/home/nikitachernysh/Storage/Projects/lidar-archaeology-segmentation/data/georef/DEM.tif"   # путь к твоему DEM GeoTIFF


In [ ]:

# Радиус анализа в метрах.
# Подбери под масштаб объектов:
# 5-10 м -> мелкие объекты
# 10-30 м -> более крупные формы
radius_m = 10.0

# Количество направлений для SVF / PO
svf_n_dir = 16

# Подавление шума:
# 0 - без подавления
# 1 - low
# 2 - medium
# 3 - high
svf_noise = 0

# Единицы slope: "degree" или "radian"
slope_units = "degree"


# =========================
# HELPERS
# =========================
def read_dem(path: str):
    with rasterio.open(path) as src:
        dem = src.read(1).astype(np.float32)
        profile = src.profile.copy()
        transform = src.transform
        nodata = src.nodata

        # размер пикселя
        res_x = abs(transform.a)
        res_y = abs(transform.e)

    if nodata is not None:
        dem = np.where(dem == nodata, np.nan, dem)

    return dem, profile, nodata, res_x, res_y


def save_raster(path: str, array: np.ndarray, profile: dict, nodata_value=np.nan):
    out_profile = profile.copy()
    out_profile.update(
        dtype="float32",
        count=1,
        compress="deflate",
        predictor=3
    )

    arr = array.astype(np.float32)

    if np.isnan(nodata_value):
        out_profile["nodata"] = np.nan
    else:
        out_profile["nodata"] = float(nodata_value)
        arr = np.where(np.isnan(arr), nodata_value, arr)

    with rasterio.open(path, "w", **out_profile) as dst:
        dst.write(arr, 1)


def compute_slope(dem: np.ndarray, res_x: float, res_y: float, units: str = "degree"):
    """
    Вычисление slope через центральные разности.
    Возвращает slope в градусах или радианах.
    """
    valid = np.isfinite(dem)
    filled = np.where(valid, dem, np.nan)

    dz_dy, dz_dx = np.gradient(filled, res_y, res_x)
    slope_rad = np.arctan(np.sqrt(dz_dx ** 2 + dz_dy ** 2))

    if units == "degree":
        slope = np.degrees(slope_rad)
    elif units == "radian":
        slope = slope_rad
    else:
        raise ValueError("units must be 'degree' or 'radian'")

    slope[~valid] = np.nan
    return slope.astype(np.float32)


# =========================
# MAIN
# =========================
dem, profile, nodata, res_x, res_y = read_dem(dem_path)

if not np.isclose(res_x, res_y):
    raise ValueError(
        f"DEM has non-square pixels: res_x={res_x}, res_y={res_y}. "
        "Для RVT и анализа рельефа лучше использовать квадратный пиксель."
    )

# RVT ожидает радиус в пикселях
radius_px = max(1, int(round(radius_m / res_x)))

# 1) SLOPE
slope = compute_slope(dem, res_x, res_y, units=slope_units)

# 2) SVF + POSITIVE OPENNESS
# В RVT SVF, anisotropic SVF и positive openness считаются одной функцией.
svf_dict = rvt.vis.sky_view_factor(
    dem=dem,
    resolution=res_x,
    compute_svf=True,
    compute_asvf=False,
    compute_opns=True,
    svf_n_dir=svf_n_dir,
    svf_r_max=radius_px,
    svf_noise=svf_noise,
    no_data=np.nan
)

svf = svf_dict["svf"].astype(np.float32)
po = svf_dict["opns"].astype(np.float32)

# 3) SAVE
out_dir = Path(dem_path).resolve().parent
save_raster(str(out_dir / "slope.tif"), slope, profile)
save_raster(str(out_dir / "svf.tif"), svf, profile)
save_raster(str(out_dir / "positive_openness.tif"), po, profile)

print("Done.")
print(f"Input DEM: {dem_path}")
print(f"Resolution: {res_x} m/pixel")
print(f"Radius: {radius_m} m -> {radius_px} px")
print("Saved:")
print(f"  {out_dir / 'slope.tif'}")
print(f"  {out_dir / 'svf.tif'}")
print(f"  {out_dir / 'positive_openness.tif'}")

In [3]:
import rasterio

with rasterio.open(dem_path) as src:
    print("CRS:", src.crs)
    print("Is projected:", src.crs.is_projected)
    print("Linear units:", src.crs.linear_units)
    print("Transform:", src.transform)

CRS: EPSG:32616
Is projected: True
Linear units: metre
Transform: | 1.00, 0.00, 215923.50|
| 0.00,-1.00, 1933132.50|
| 0.00, 0.00, 1.00|
